# Phase 3 — Corruption Benchmark (Colab T4)

**Pre-requisite:** Phase 2 done — `models/baseline/best.pt` in Drive, clean test set present.

Produces the robustness profile: how the baseline model degrades under 25 corruption
scenarios (5 types x 5 severities, ImageNet-C methodology).

**Phase 3 is pure measurement — no target.** Record whatever mPC comes out.
If relative mPC >= 0.85 the model is suspiciously robust — recheck corruption code.

In [ ]:
import os, json, shutil, tempfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR  = '/content/drive/MyDrive/robot-perception'
RESULTS_DIR  = f'{PROJECT_DIR}/results/figures'
RESULTS_JSON = f'{PROJECT_DIR}/results/robustness_baseline.json'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
CORRUPTIONS = ['gaussian_noise', 'motion_blur', 'gaussian_blur', 'brightness', 'occlusion']
SEVERITIES  = [1, 2, 3, 4, 5]

## Step 1 — Copy clean test set to local disk

Drive I/O is slow. Corruptions are generated on local disk.

In [ ]:
LOCAL_DATA = '/content/robot_data'
if not os.path.exists(LOCAL_DATA):
    shutil.copytree(f'{PROJECT_DIR}/data/annotated', LOCAL_DATA)

LOCAL_TEST_IMG = f'{LOCAL_DATA}/images/test'
LOCAL_TEST_LBL = f'{LOCAL_DATA}/labels/test'

test_images = sorted(
    os.path.join(LOCAL_TEST_IMG, f) for f in os.listdir(LOCAL_TEST_IMG)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
)
print(f'Clean test set: {len(test_images)} images')

## Step 2 — Corruption function

Identical to `scripts/generate_corruptions.py`. Severity formulas are the
authoritative ImageNet-C-style values from the project plan.

In [ ]:
def apply_corruption(image, corruption_type, severity):
    """Apply one corruption at one severity. severity: 1=mild, 5=severe."""
    s = severity

    if corruption_type == 'gaussian_noise':
        std = [0.04, 0.08, 0.12, 0.18, 0.26][s-1]
        noise = np.random.normal(0, std * 255, image.shape).astype(np.int16)
        return np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    elif corruption_type == 'motion_blur':
        k = [3, 5, 7, 11, 15][s-1]
        kernel = np.zeros((k, k))
        kernel[k//2, :] = 1.0 / k
        return cv2.filter2D(image, -1, kernel)

    elif corruption_type == 'gaussian_blur':
        sigma = [0.5, 1.0, 1.5, 2.5, 4.0][s-1]
        k = int(6*sigma+1) | 1
        return cv2.GaussianBlur(image, (k, k), sigma)

    elif corruption_type == 'brightness':
        factor = [1.3, 1.6, 2.0, 2.5, 3.0][s-1]
        return np.clip(image.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    elif corruption_type == 'occlusion':
        h, w = image.shape[:2]
        result = image.copy()
        n_patches = [1, 2, 3, 5, 8][s-1]
        rng = np.random.default_rng(seed=42)  # deterministic benchmark
        for _ in range(n_patches):
            pw, ph = int(w*0.08), int(h*0.08)
            x = rng.integers(0, max(1, w - pw))
            y = rng.integers(0, max(1, h - ph))
            result[y:y+ph, x:x+pw] = 0
        return result

    return image

## VIZ 3.A — Corruption strip

**Run before generating or evaluating anything.** Visually confirm severities look
mild→severe and realistic.

In [ ]:
sample_img = cv2.cvtColor(cv2.imread(test_images[0]), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(len(CORRUPTIONS), len(SEVERITIES) + 1,
                          figsize=(18, len(CORRUPTIONS) * 3))
for i, corruption in enumerate(CORRUPTIONS):
    axes[i, 0].imshow(sample_img)
    axes[i, 0].set_title('Original' if i == 0 else '', fontsize=9)
    axes[i, 0].set_ylabel(corruption, fontsize=10, fontweight='bold')
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    for j, severity in enumerate(SEVERITIES):
        corrupted = apply_corruption(sample_img.copy(), corruption, severity)
        axes[i, j+1].imshow(np.clip(corrupted, 0, 255).astype(np.uint8))
        axes[i, j+1].set_title(f'S{severity}' if i == 0 else '', fontsize=9)
        axes[i, j+1].axis('off')

plt.suptitle('VIZ 3.A — Corruption strip: all types x all severities', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz3a_corruption_strip.png', dpi=120)
plt.show()
print('⚠️  MANUAL CHECK: S1 mild? S5 severe? Occlusion = realistic blocking?')
print('   If any severity looks wrong, fix apply_corruption before continuing.')

## Step 3 — Generate all 25 corrupted test sets

Layout `severity_{N}/images/` + `severity_{N}/labels/` is required so Ultralytics
can find labels (it swaps `/images/` -> `/labels/` in image paths).

In [ ]:
CORRUPTED_LOCAL = '/content/corrupted'
total = len(CORRUPTIONS) * len(SEVERITIES)
done = 0

for corruption in CORRUPTIONS:
    for severity in SEVERITIES:
        sev_dir  = f'{CORRUPTED_LOCAL}/{corruption}/severity_{severity}'
        img_out  = f'{sev_dir}/images'
        lbl_out  = f'{sev_dir}/labels'
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(lbl_out, exist_ok=True)

        for img_path in test_images:
            name = os.path.basename(img_path)
            stem = os.path.splitext(name)[0]
            dest = f'{img_out}/{name}'
            if not os.path.exists(dest):
                img = cv2.imread(img_path)
                corrupted = apply_corruption(img, corruption, severity)
                cv2.imwrite(dest, corrupted)
            # copy shared test label
            lbl_src = f'{LOCAL_TEST_LBL}/{stem}.txt'
            lbl_dst = f'{lbl_out}/{stem}.txt'
            if os.path.exists(lbl_src) and not os.path.exists(lbl_dst):
                shutil.copy2(lbl_src, lbl_dst)

        done += 1
        print(f'  [{done}/{total}] {corruption}/severity_{severity}')

print('\nAll 25 corrupted sets generated.')

In [ ]:
# OPTIONAL — persist corrupted sets to Drive so Phase 4 can reuse them
# without regenerating. Recommended (Phase 4 evaluates the robust model on
# the SAME 25 sets). Skip if Drive storage is tight — they are regenerable.
DRIVE_CORRUPTED = f'{PROJECT_DIR}/data/corrupted'
if not os.path.exists(DRIVE_CORRUPTED):
    shutil.copytree(CORRUPTED_LOCAL, DRIVE_CORRUPTED)
    print(f'Persisted to {DRIVE_CORRUPTED}')
else:
    print('Drive copy already exists — skipping.')

## Step 4 — Evaluate baseline model on all 25 sets

In [ ]:
from ultralytics import RTDETR

model = RTDETR(f'{PROJECT_DIR}/models/baseline/best.pt')

def build_temp_yaml(sev_dir, tmp_dir):
    """Yaml whose path is the severity dir; test='images' -> labels found at images-swap."""
    content = f"""path: {os.path.abspath(sev_dir)}
train: images
val: images
test: images

nc: 5
names: ['arm', 'leg', 'torso', 'head', 'sensor']
"""
    p = os.path.join(tmp_dir, os.path.basename(sev_dir) + '.yaml')
    with open(p, 'w') as f:
        f.write(content)
    return p

In [ ]:
# Clean test set first
import glob
LOCAL_YAML = '/content/robot_parts.yaml'
with open(LOCAL_YAML, 'w') as f:
    f.write(f"""path: {LOCAL_DATA}
train: images/train
val: images/val
test: images/test

nc: 5
names: ['arm', 'leg', 'torso', 'head', 'sensor']
""")

clean_metrics = model.val(data=LOCAL_YAML, split='test', verbose=False)
clean_mAP = float(clean_metrics.box.map50)
print(f'Clean mAP@0.5: {clean_mAP:.4f}')

# All 25 corrupted sets
results_grid = {}
tmp_dir = tempfile.mkdtemp()
done = 0

for corruption in CORRUPTIONS:
    results_grid[corruption] = {}
    for severity in SEVERITIES:
        sev_dir = f'{CORRUPTED_LOCAL}/{corruption}/severity_{severity}'
        # clear stale Ultralytics label cache
        for c in glob.glob(f'{sev_dir}/labels/*.cache'):
            os.remove(c)
        yaml_path = build_temp_yaml(sev_dir, tmp_dir)
        try:
            m = model.val(data=yaml_path, split='test', verbose=False)
            mAP = float(m.box.map50)
        except Exception as e:
            print(f'  ERROR {corruption}/s{severity}: {e}')
            mAP = 0.0
        results_grid[corruption][severity] = mAP
        done += 1
        print(f'  [{done}/25] {corruption}/severity_{severity}: mAP={mAP:.4f}')

## Step 5 — Compute mPC and save JSON

In [ ]:
all_mAPs = [results_grid[c][s] for c in CORRUPTIONS for s in SEVERITIES]
mPC = float(np.mean(all_mAPs))
relative_mPC = mPC / clean_mAP if clean_mAP > 0 else 0.0

output = {
    'weights': f'{PROJECT_DIR}/models/baseline/best.pt',
    'clean_mAP50': clean_mAP,
    'mPC': mPC,
    'relative_mPC': relative_mPC,
    'results_grid': {c: {str(s): results_grid[c][s] for s in SEVERITIES}
                     for c in CORRUPTIONS},
}
with open(RESULTS_JSON, 'w') as f:
    json.dump(output, f, indent=2)

print('=== Robustness Summary (baseline) ===')
print(f'Clean mAP@0.5:  {clean_mAP:.4f}')
print(f'mPC:            {mPC:.4f}')
print(f'Relative mPC:   {relative_mPC:.4f}  ({relative_mPC*100:.1f}% of clean retained)')

per_corruption = {c: np.mean([results_grid[c][s] for s in SEVERITIES]) for c in CORRUPTIONS}
worst = min(per_corruption, key=per_corruption.get)
print(f'Worst corruption: {worst} (avg mAP={per_corruption[worst]:.4f})')
s5_avg = np.mean([results_grid[c][5] for c in CORRUPTIONS])
print(f'Avg mAP at severity 5: {s5_avg:.4f}')
print(f'\nSaved: {RESULTS_JSON}')
if relative_mPC >= 0.85:
    print('⚠️  Relative mPC >= 0.85 — model suspiciously robust. Recheck corruption code.')

## Step 6 — Robustness curve + drop heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for corruption in CORRUPTIONS:
    mAPs = [results_grid[corruption][s] for s in SEVERITIES]
    ax.plot(SEVERITIES, mAPs, marker='o', label=corruption)
ax.axhline(y=clean_mAP, color='k', linestyle='--', label='clean baseline')
ax.set_xlabel('Corruption Severity'); ax.set_ylabel('mAP@0.5')
ax.set_title('Robustness Curve — Baseline Model')
ax.set_xticks(SEVERITIES); ax.legend()

drops = np.array([[clean_mAP - results_grid[c][s] for s in SEVERITIES] for c in CORRUPTIONS])
ax = axes[1]
im = ax.imshow(drops, cmap='Reds', aspect='auto')
ax.set_xticks(range(5)); ax.set_xticklabels([f'S{i}' for i in SEVERITIES])
ax.set_yticks(range(len(CORRUPTIONS))); ax.set_yticklabels(CORRUPTIONS)
ax.set_title('mAP Drop Heatmap (redder = worse)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/robustness_curve_baseline.png', dpi=150)
plt.show()

## Phase 3 Completion Checklist

- [ ] All 5 corruption types implemented
- [ ] VIZ 3.A saved — severities look realistic and progressive
- [ ] All 25 corrupted sets generated (images/ + labels/ layout)
- [ ] Baseline model evaluated on all 25 sets
- [ ] `results/robustness_baseline.json` saved
- [ ] robustness_curve_baseline.png saved
- [ ] mPC computed and recorded

**Sanity check:** during `model.val()` the printed summary must show a non-zero
instance count. If instances = 0, the label path layout is wrong.

Next → `04_robust_training.ipynb`